In [ ]:
import os
import time
import random
import shutil
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR


def setup_datasets(base_dir="./mri_project_data"):
    """
    Downloads raw brain MRI data and partitions it to simulate cross-dataset domain shift.
    BRISC split: 80% train, 20% validation.
    Figshare split: Held-out test set for out-of-distribution evaluation.
    """
    repo_url = "https://github.com/SartajBhuvaji/Brain-Tumor-Classification-DataSet.git"
    clone_dir = "Brain-Tumor-Classification-DataSet"
    classes = ['glioma_tumor', 'meningioma_tumor', 'pituitary_tumor', 'no_tumor']

    if not os.path.exists(clone_dir):
        print("Fetching primary MRI dataset from repository...")
        subprocess.run(["git", "clone", repo_url], check=True)

    splits = ['brisc_train', 'brisc_val', 'figshare_test']
    for split in splits:
        for cls in classes:
            os.makedirs(os.path.join(base_dir, split, cls), exist_ok=True)

    print("Partitioning images into training, validation, and cross-dataset test sets...")
    for cls in classes:
        src_train_dir = os.path.join(clone_dir, 'Training', cls)
        src_test_dir = os.path.join(clone_dir, 'Testing', cls)

        train_imgs = os.listdir(src_train_dir)
        random.seed(42)
        random.shuffle(train_imgs)

        split_idx = int(len(train_imgs) * 0.8)
        brisc_train = train_imgs[:split_idx]
        brisc_val = train_imgs[split_idx:]

        for img in brisc_train:
            shutil.copy(os.path.join(src_train_dir, img), os.path.join(base_dir, 'brisc_train', cls, img))

        for img in brisc_val:
            shutil.copy(os.path.join(src_train_dir, img), os.path.join(base_dir, 'brisc_val', cls, img))

        for img in os.listdir(src_test_dir):
            shutil.copy(os.path.join(src_test_dir, img), os.path.join(base_dir, 'figshare_test', cls, img))

    return classes


DATA_DIR = "./mri_project_data"
CLASSES = setup_datasets(DATA_DIR)
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

imagenet_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        imagenet_normalize
    ]),
    'eval': transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        imagenet_normalize
    ])
}

image_datasets = {
    'brisc_train': datasets.ImageFolder(os.path.join(DATA_DIR, 'brisc_train'), data_transforms['train']),
    'brisc_val': datasets.ImageFolder(os.path.join(DATA_DIR, 'brisc_val'), data_transforms['eval']),
    'figshare_test': datasets.ImageFolder(os.path.join(DATA_DIR, 'figshare_test'), data_transforms['eval'])
}

dataloaders = {
    'brisc_train': DataLoader(image_datasets['brisc_train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
    'brisc_val': DataLoader(image_datasets['brisc_val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
    'figshare_test': DataLoader(image_datasets['figshare_test'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
}


class ChannelAttention(nn.Module):
    """
    Channel Attention Module (CBAM)
    Squeezes spatial dimensions via Average and Max pooling to compute relative channel weights.
    """
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, kernel_size=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    """
    Spatial Attention Module (CBAM)
    Aggregates channel information via mean and max pooling to generate 2D spatial focus maps.
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat))


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module combining sequential channel and spatial attention filters.
    """
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x


class TBACNN(nn.Module):
    """
    Tri-Branch Attention CNN Architecture.
    Combines localized texture extraction (Branch 1) with dilated global context extraction (Branch 2),
    fused via CBAM attention refinement.
    """
    def __init__(self, num_classes=4):
        super().__init__()

        # Branch 1: Standard Convolutions for localized edge and texture extraction
        self.branch1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )

        # Branch 2: Dilated Convolutions (rate=2) to expand receptive field for macro spatial context
        self.branch2 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=2, dilation=2), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=2, dilation=2), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=2, dilation=2), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )

        # Dual-branch fusion layer (64 channels + 64 channels = 128 channels)
        self.fusion_cbam = CBAM(in_planes=128)
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        b1_feats = self.branch1(x)
        b2_feats = self.branch2(x)

        fused_feats = torch.cat((b1_feats, b2_feats), dim=1)
        attended_feats = self.fusion_cbam(fused_feats)

        pooled = self.global_avg_pool(attended_feats).view(x.size(0), -1)
        return self.classifier(pooled)


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = TBACNN(num_classes=len(CLASSES)).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model initialized: TBA-CNN architecture ({trainable_params:,} trainable parameters).")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = StepLR(optimizer, step_size=3, gamma=0.1)

EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"Starting model optimization on device: {device}")
for epoch in range(EPOCHS):
    start_time = time.time()

    for phase in ['brisc_train', 'brisc_val']:
        model.train() if phase == 'brisc_train' else model.eval()
        running_loss, running_corrects = 0.0, 0

        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'brisc_train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'brisc_train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        if phase == 'brisc_train':
            scheduler.step()

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = (running_corrects.double() / len(image_datasets[phase])).item()

        log_key = 'train' if phase == 'brisc_train' else 'val'
        history[f'{log_key}_loss'].append(epoch_loss)
        history[f'{log_key}_acc'].append(epoch_acc)

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Duration: {epoch_time:.1f}s | "
          f"Train Loss: {history['train_loss'][-1]:.4f} | Train Acc: {history['train_acc'][-1]:.4f} | "
          f"Val Loss: {history['val_loss'][-1]:.4f} | Val Acc: {history['val_acc'][-1]:.4f}")

torch.save(model.state_dict(), 'tba_cnn_brain_tumor.pth')
print("Model weights successfully serialized to 'tba_cnn_brain_tumor.pth'.")

print("\nEvaluating zero-shot cross-dataset performance on Figshare test set...")
model.load_state_dict(torch.load('tba_cnn_brain_tumor.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in dataloaders['figshare_test']:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

metrics_dict = classification_report(all_labels, all_preds, target_names=CLASSES, output_dict=True, zero_division=0)
metrics_df = pd.DataFrame(metrics_dict).transpose()
metrics_df.to_csv("classification_report.csv")
print("Classification metrics saved to 'classification_report.csv'.")

sns.set_theme(style="whitegrid")

# Figure 1: Loss & Accuracy Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_title('Cross-Entropy Loss Trajectory', fontweight='semibold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history['train_acc'], label='Train Accuracy', marker='o')
ax2.plot(history['val_acc'], label='Val Accuracy', marker='s')
ax2.set_title('Classification Accuracy Trajectory', fontweight='semibold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300)
plt.close()

# Figure 2: Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[c.replace('_tumor', '') for c in CLASSES],
            yticklabels=[c.replace('_tumor', '') for c in CLASSES])
plt.title('Cross-Dataset Confusion Matrix (Figshare Evaluation)', fontweight='semibold')
plt.ylabel('Ground Truth')
plt.xlabel('Model Prediction')

f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
rec = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
metrics_text = f"Weighted F1-Score: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}"
plt.figtext(0.5, -0.05, metrics_text, wrap=True, horizontalalignment='center', fontsize=11, fontweight='bold')

plt.savefig('cross_dataset_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

print("Generated visual plots: 'training_curves.png' and 'cross_dataset_confusion_matrix.png'.")



Fetching primary MRI dataset from repository...
Partitioning images into training, validation, and cross-dataset test sets...
Model initialized: TBA-CNN architecture (50,278 trainable parameters).
Starting model optimization on device: cuda:0
Epoch 01/10 | Duration: 13.9s | Train Loss: 1.1224 | Train Acc: 0.5183 | Val Loss: 1.3883 | Val Acc: 0.3646
Epoch 02/10 | Duration: 14.3s | Train Loss: 0.9612 | Train Acc: 0.5976 | Val Loss: 0.9087 | Val Acc: 0.6372
Epoch 03/10 | Duration: 13.8s | Train Loss: 0.9076 | Train Acc: 0.6116 | Val Loss: 3.0608 | Val Acc: 0.3767
Epoch 04/10 | Duration: 13.9s | Train Loss: 0.8278 | Train Acc: 0.6639 | Val Loss: 0.7452 | Val Acc: 0.6927
Epoch 05/10 | Duration: 14.0s | Train Loss: 0.7988 | Train Acc: 0.6774 | Val Loss: 0.7178 | Val Acc: 0.6962
Epoch 06/10 | Duration: 13.8s | Train Loss: 0.7686 | Train Acc: 0.6800 | Val Loss: 0.7247 | Val Acc: 0.6823
Epoch 07/10 | Duration: 13.9s | Train Loss: 0.7701 | Train Acc: 0.6809 | Val Loss: 0.7132 | Val Acc: 0.6944
E

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>